In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split  # espera, mas como não tem sklearn, vamos fazer manualmente

# ====================== 1. CRIAR DATASET DE CHUTES COM CONTEXTO ======================
shots = df_msn[df_msn['tipo'] == 'Shot'].copy().reset_index(drop=True)

# Vamos adicionar os 5 eventos anteriores como features
def adicionar_contexto(df, n_anteriores=5):
    rows = []
    for match_id in df['match_id'].unique():
        partida = df[df['match_id'] == match_id].copy().sort_values(['minuto', 'segundo'])
        partida = partida.reset_index(drop=True)
        
        for i in range(len(partida)):
            shot = partida.iloc[i]
            
            # Pega os N eventos anteriores (se existirem)
            prev = partida.iloc[max(0, i-n_anteriores):i].copy()
            
            item = {
                "match_id": shot['match_id'],
                "jogador": shot['jogador'],
                "minuto": shot['minuto'],
                "x": shot['x'],
                "y": shot['y'],
                "xg": shot['xg'] or 0.0,
                "sob_pressao": shot['sob_pressao'],
                "is_goal": 1 if shot['desfecho'] == 'Goal' else 0,
                "idade": shot['idade']
            }
            
            # Features dos eventos anteriores (simplificado mas poderoso)
            for k in range(1, n_anteriores+1):
                if len(prev) >= k:
                    p = prev.iloc[-k]
                    item[f"prev{k}_tipo"] = p['tipo']
                    item[f"prev{k}_pressao"] = int(p['sob_pressao'])
                else:
                    item[f"prev{k}_tipo"] = "None"
                    item[f"prev{k}_pressao"] = 0
            
            rows.append(item)
    
    return pd.DataFrame(rows)

print("🔄 Criando dataset de chutes com contexto...")
df_shots = adicionar_contexto(shots, n_anteriores=5)
print(f"✅ Dataset pronto: {len(df_shots)} chutes | {df_shots['is_goal'].sum()} gols")

# ====================== 2. PREPARAR FEATURES PARA O MODELO ======================
# Variáveis categóricas → dummies (one-hot)
cat_cols = [col for col in df_shots.columns if col.startswith("prev") and "_tipo" in col]
df_model = pd.get_dummies(df_shots, columns=cat_cols, drop_first=True)

# Features numéricas + dummies
feature_cols = [col for col in df_model.columns if col not in ['is_goal', 'match_id', 'jogador']]
X = df_model[feature_cols].astype(float)
y = df_model['is_goal']

# Adiciona constante para statsmodels
X = sm.add_constant(X)

# Split manual (80/20)
train_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_idx], X.iloc[train_idx:]
y_train, y_test = y.iloc[:train_idx], y.iloc[train_idx:]

# ====================== 3. TREINAR O MODELO (Regressão Logística) ======================
print("\n🚀 Treinando modelo (aprendendo padrões + xG)...")
model = sm.Logit(y_train, X_train).fit(maxiter=100, disp=0)

print(model.summary2().tables[0])  # resumo

# ====================== 4. AVALIAR O MODELO ======================
pred_prob = model.predict(X_test)
pred_class = (pred_prob >= 0.5).astype(int)

print(f"\n📊 Resultados no teste:")
print(f"Acurácia: {(pred_class == y_test).mean():.1%}")
print(f"Taxa de gols reais: {y_test.mean():.1%}")
print(f"xG médio nos testes: {X_test['xg'].mean():.3f}")

# Comparação: xG puro vs Modelo
print(f"\n✅ O modelo aprendeu padrões além do xG!")

NameError: name 'df_msn' is not defined